In [1]:
import pandas as pd
import re

In [9]:
CSV_FILE = r'C:\Users\RTX\Documents\GitHub\PT\PROYECTO_TERMINAL_INVENTARIO_TI\datos\DEPARTAMENTOS_1.xlsx'
OUT_DIR = r'C:\Users\RTX\Documents\GitHub\PT\PROYECTO_TERMINAL_INVENTARIO_TI\OUT'


In [30]:

raw_df = pd.read_excel(CSV_FILE, sheet_name=None)



In [36]:
raw_df

{'ISLA17(42)':                SITIO           TIENDA  ESTACION
 0        BEACHPALACE       TABAQUERIA  SERVIDOR
 1        BEACHPALACE       TABAQUERIA      CAJA
 2        BEACHPALACE          JOYERIA      CAJA
 3        *CUN T2 PA            CORONA  SERVIDOR
 4        *CUN T2 PB            CORONA  SERVIDOR
 5    CUN T3 ISLA C11           CORONA  SERVIDOR
 6    CUN T3 ISLA C20           CORONA  SERVIDOR
 7    LE BLANC CANCUN       TABAQUERIA  SERVIDOR
 8    LE BLANC CANCUN       TABAQUERIA      CAJA
 9    LE BLANC CANCUN          JOYERIA      CAJA
 10        MOON GRAND            MULTI  SERVIDOR
 11        MOON GRAND            MULTI      CAJA
 12        MOON GRAND         PUEBLITO      CAJA
 13        MOON GRAND         PUEBLITO      CAJA
 14        MOON GRAND          JOYERIA      CAJA
 15        MOON GRAND       RECEPTORIA       NaN
 16        MOON GRAND           AMIANI  SERVIDOR
 17        MOON GRAND       AQUAPARQUE  SERVIDOR
 18        MOON GRAND   PIER27 PERUANO  SERVIDOR
 19   

In [17]:
def limpieza(texto):
    if pd.isna(texto) or not isinstance(texto, str):
        return texto
    texto = texto.upper()
    texto = texto.replace('RECEPTOR\x8dA', 'RECEPTORIA')
    texto = re.sub(r'[^A-Z0-9\s]', '', texto)
    return texto.strip()

for col in raw_df.columns:
    if raw_df[col].dtype == 'object':
        raw_df[col] = raw_df[col].apply(limpieza)

util_df = raw_df.copy()
print(f"✅ Limpieza aplicada")

✅ Limpieza aplicada


In [18]:
EMPRESA_CONFIG = {
    'ISLA17':    r'(TAB|JOY|CORONA|MULTI|PUEBLITO|RECEPTORA|AMIANI|AQUA PARK|PROSHOP|PIER 27|PIER27|CORAZON)',
    'HSPRO':     r'(TAB|JOY|RECEPTOR|AMIANI|PIER27|TABAQUERIA|BOUTIQUE|VIVID|LOBBY|WHITESANDS)',
    'PTOARENAS': r'(TAB|JOY|CORONA|MULTI|PUEBLITO|RECEPTORIA|AMIANI|AQUA PARK|PROSHOP|PIER 27|PIER27|CORAZON|VIVA|MARINI|SUNGLASSCHIC|ALBERCA|MUL|RECEPTOR)',
    'PTOHS':     r'(TAB|JOY|CORONA|MULTI|PUEBLITO|RECEPTORIA|AMIANI|AQUA PARK|PROSHOP|PIER 27|PIER27|CORAZON|VIVA|MARINI|SUNGLASSCHIC|ALBERCA|MUL|RECEPTOR|BOARWALK|BOARDWALK|FINI|CORAZN|BODEGA)',
    'PTO85':     r'(TAB|JOY|CORONA|MULTI|PUEBLITO|RECEPTORIA|AMIANI|AQUA PARK|PROSHOP|PIER 27|PIER27|CORAZON|VIVA|MARINI|SUNGLASSCHIC|ALBERCA|MUL|RECEPTOR|BOARWALK|BOARDWALK|FINI|CORAZN|BODEGA|LOS CABOS|CABOS|ALMACEN)',
    'PTOHSRD':   r'(TAB|JOY|CORONA|MULTI|PUEBLITO|RECEPTORIA|AMIANI|AQUA PARK|PROSHOP|PIER 27|PIER27|CORAZON|VIVA|MARINI|SUNGLASSCHIC|ALBERCA|MUL|RECEPTOR|BOARWALK|BOARDWALK|FINI|CORAZN|BODEGA|LOS CABOS|CABOS|ALMACEN|RIDDIM)',
    'PELTJAM':   r'(CORAL SPRINGS|OCHO RIOS|CEDIS|TAB|JOY|CORONA|MULTI|PUEBLITO|RECEPTORIA|AMIANI|AQUA PARK|PROSHOP|PIER 27|PIER27|CORAZON|VIVA|MARINI|SUNGLASSCHIC|ALBERCA|MUL|RECEPTOR|BOARWALK|BOARDWALK|FINI|CORAZN|BODEGA|LOS CABOS|CABOS|ALMACEN|RIDDIM)',
    'OHSGRLTD':  r'(CORAL SPRINGS|OCHO RIOS|CEDIS|TAB|JOY|CORONA|MULTI|PUEBLITO|RECEPTORIA|AMIANI|AQUA PARK|PROSHOP|PIER 27|PIER27|CORAZON|VIVA|MARINI|SUNGLASSCHIC|ALBERCA|MUL|RECEPTOR|BOARWALK|BOARDWALK|FINI|CORAZN|BODEGA|LOS CABOS|CABOS|ALMACEN|RIDDIM)',
    'OHSXCD':    r'(CORAL SPRINGS|OCHO RIOS|CEDIS|TAB|JOY|CORONA|MULTI|PUEBLITO|RECEPTORIA|AMIANI|AQUA PARK|PROSHOP|PIER 27|PIER27|CORAZON|VIVA|MARINI|SUNGLASSCHIC|ALBERCA|MUL|RECEPTOR|BOARWALK|BOARDWALK|FINI|CORAZN|BODEGA|LOS CABOS|CABOS|ALMACEN|RIDDIM)',
}

partes = []

for empresa, keywords in EMPRESA_CONFIG.items():
    df = util_df[util_df['RAZON_SOCIAL'] == empresa].copy()
    if df.empty:
        print(f"  ⚠️  {empresa}: sin registros en el CSV")
        continue
    df['HOTEL']       = df['TIENDAS'].str.split(keywords).str[0].str.strip()
    df['HOTEL']       = df['HOTEL'].str.replace(r'^[^\w]+', '', regex=True)
    df['TIPO_EQUIPO'] = df['TIENDAS'].str.extract(keywords, expand=False).fillna('OTRO')
    partes.append(df)
    print(f"  ✅ {empresa}: {len(df)} filas procesadas")

final_df = pd.concat(partes, ignore_index=True)
print(f"\n✅ Total transformado: {len(final_df)} filas")


  ✅ ISLA17: 43 filas procesadas
  ✅ HSPRO: 56 filas procesadas
  ✅ PTOARENAS: 48 filas procesadas
  ✅ PTOHS: 67 filas procesadas
  ✅ PTO85: 33 filas procesadas
  ✅ PTOHSRD: 40 filas procesadas
  ✅ PELTJAM: 41 filas procesadas
  ✅ OHSGRLTD: 3 filas procesadas
  ✅ OHSXCD: 4 filas procesadas

✅ Total transformado: 335 filas


In [19]:
def extraer_vendor(modelo):
    if pd.isna(modelo) or str(modelo).strip() == '':
        return 'DESCONOCIDO'
    return str(modelo).split()[0]

final_df['VENDOR_NOMBRE'] = final_df['MODELO'].apply(extraer_vendor)

In [20]:
# --- dim_company ---
dim_company = (
    final_df[['RAZON_SOCIAL']]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_company.insert(0, 'id', dim_company.index + 1)
dim_company.columns = ['id', 'companyName']

# --- dim_site ---
dim_site = (
    final_df[['HOTEL', 'RAZON_SOCIAL']]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_site = dim_site.merge(dim_company, left_on='RAZON_SOCIAL', right_on='companyName', how='left')
dim_site = dim_site[['HOTEL', 'id']].rename(columns={'HOTEL': 'siteName', 'id': 'companyId'})
dim_site = dim_site[dim_site['siteName'].notna() & (dim_site['siteName'] != '')]
dim_site.insert(0, 'id', range(1, len(dim_site) + 1))

# --- dim_product_type ---
dim_product_type = (
    final_df[['TIPO_EQUIPO']]
    .drop_duplicates()
    .reset_index(drop=True)
    .rename(columns={'TIPO_EQUIPO': 'typeName'})
)
dim_product_type.insert(0, 'id', dim_product_type.index + 1)

# --- dim_vendor ---
dim_vendor = (
    final_df[['VENDOR_NOMBRE']]
    .drop_duplicates()
    .reset_index(drop=True)
    .rename(columns={'VENDOR_NOMBRE': 'vendorName'})
)
dim_vendor.insert(0, 'id', dim_vendor.index + 1)

# --- dim_asset_detail ---
dim_asset_detail = final_df[[
    'HOSTNAME', 'NS', 'MODELO', 'SO', 'CPU', 'HHD SSD NVME', 'RAM'
]].copy().reset_index(drop=True)
dim_asset_detail.columns = ['hostname', 'serialNumber', 'modelo', 'operatingSystem', 'cpu', 'storage', 'ram']
dim_asset_detail.insert(0, 'id', dim_asset_detail.index + 1)

# --- fact_asset (tabla de hechos con todas las FK) ---
fact_asset = final_df.copy().reset_index(drop=True)
fact_asset['assetDetailId'] = dim_asset_detail['id'].values

fact_asset = fact_asset.merge(dim_company, left_on='RAZON_SOCIAL', right_on='companyName', how='left').rename(columns={'id': 'companyId'})
fact_asset = fact_asset.merge(dim_site[['id', 'siteName', 'companyId']], left_on=['HOTEL', 'companyId'], right_on=['siteName', 'companyId'], how='left').rename(columns={'id': 'siteId'})
fact_asset = fact_asset.merge(dim_product_type, left_on='TIPO_EQUIPO', right_on='typeName', how='left').rename(columns={'id': 'productTypeId'})
fact_asset = fact_asset.merge(dim_vendor, left_on='VENDOR_NOMBRE', right_on='vendorName', how='left').rename(columns={'id': 'vendorId'})

fact_asset = fact_asset[[
    'TIENDAS', 'companyId', 'siteId', 'productTypeId', 'vendorId', 'assetDetailId'
]].rename(columns={'TIENDAS': 'assetName'})
fact_asset.insert(0, 'id', fact_asset.index + 1)


In [35]:

import os
os.makedirs(OUT_DIR, exist_ok=True)

tablas = {
    'dim_company':      dim_company,
    'dim_site':         dim_site,
    'dim_product_type': dim_product_type,
    'dim_vendor':       dim_vendor,
    'dim_asset_detail': dim_asset_detail,
    'fact_asset':       fact_asset,
}

print("\n📂 Archivos generados:")
for nombre, df in tablas.items():
    ruta = f"{OUT_DIR}{nombre}.csv"
    df.to_csv(ruta, index=False, encoding='utf-8-sig')  # utf-8-sig para que Excel no truene
    print(f"  ✅ {nombre}.csv — {len(df)} filas, {len(df.columns)} columnas")




📂 Archivos generados:
  ✅ dim_company.csv — 9 filas, 2 columnas
  ✅ dim_site.csv — 178 filas, 3 columnas
  ✅ dim_product_type.csv — 35 filas, 2 columnas
  ✅ dim_vendor.csv — 9 filas, 2 columnas
  ✅ dim_asset_detail.csv — 335 filas, 8 columnas
  ✅ fact_asset.csv — 335 filas, 7 columnas


In [36]:
print("\n📊 Reporte de validación:")
print(f"  Empresas únicas:       {dim_company['companyName'].nunique()}")
print(f"  Hoteles/Sites únicos:  {dim_site['siteName'].nunique()}")
print(f"  Tipos de equipo:       {dim_product_type['typeName'].nunique()}")
print(f"  Vendors detectados:    {dim_vendor['vendorName'].nunique()}")
print(f"  Total activos (Asset): {len(fact_asset)}")

nulos = fact_asset[['siteId', 'companyId', 'productTypeId', 'vendorId']].isna().sum()
if nulos.sum() > 0:
    print(f"\n  ⚠️  FKs con valor nulo (revisar antes del load real):")
    print(nulos[nulos > 0].to_string())
else:
    print(f"\n  ✅ Todas las llaves foráneas resueltas correctamente")

print("\n🎉 Dry run completado. Revisa los CSVs en:", OUT_DIR)



📊 Reporte de validación:
  Empresas únicas:       9
  Hoteles/Sites únicos:  177
  Tipos de equipo:       35
  Vendors detectados:    9
  Total activos (Asset): 335

  ⚠️  FKs con valor nulo (revisar antes del load real):
siteId    4

🎉 Dry run completado. Revisa los CSVs en: C:\Users\RTX\Documents\GitHub\PT\PROYECTO_TERMINAL_INVENTARIO_TI\OUT
